# VBZ Event-Kalender
### Strukturierter Event-Kalender Zürich 2023–2025 als Feature für die Verspätungsanalyse


Aufbau eines strukturierten Event-Kalenders für Zürich (2023–2025), der als Feature
für die Verspätungsanalyse und das ML-Modell genutzt wird.

**Schwellwert:** Nur Events mit voraussichtlich >1.000 Besuchern werden berücksichtigt —
kleinere Veranstaltungen haben keinen messbaren Einfluss auf das Tramnetz.

**Gewichtung:** Der erwartete Besucheransturm wird in drei Stufen abgebildet.
Die Besucherzahlen beziehen sich auf tatsächlich anwesende Personen, nicht auf
verkaufte Tickets — gerade beim Fussball weicht das deutlich ab, da viele
Dauerkarteninhaber nicht zu jedem Heimspiel erscheinen.

| Gewichtung | Bedeutung | Besucher (real) | Beispiele |
| :--- | :--- | :--- | :--- |
| `3` — Sehr hoch | Massen-Event, stadtweite Auswirkung | >30.000 | Street Parade (~1 Mio.), Züri Fäscht (~2 Mio. über 3 Tage), Taylor Swift (~40.000/Abend), AC/DC (~40.000) |
| `2` — Hoch | Grosser Event, lokaler Impact | 10.000–30.000 | FCZ Super League (~11.500 real), Silvesterzauber (~50.000), Auto Zürich (~10.000/Tag), Ed Sheeran (~35.000) |
| `1` — Mittel | Mittlerer Event | 1.000–10.000 | GCZ Super League (~5.000–8.000 real), Schweizer Cup, Kongresse, Fachmessen, Feiertage |

> **Hinweis Fussball:** Die offiziellen Zuschauerzahlen der Clubs weichen stark von
> den realen ab. FCZ meldet im Schnitt ~15.000, tatsächlich kommen ~11.500.
> Ursache: Dauerkarteninhaber werden mitgezählt, erscheinen aber nicht immer.
> Quelle: Tages-Anzeiger, September 2025.

**Datenquellen:**

| Kategorie | Quelle | Methode |
| :--- | :--- | :--- |
| Feiertage | Python `holidays`-Package | Automatisch für Kanton Zürich generiert |
| Stadtfeste | Gemini | Grösste Zürcher Stadtfeste recherchiert |
| Konzerte | Perplexity | Grosskonzerte >1.000 Besucher im Letzigrund |
| Fachmessen | Perplexity | Grösste Messen in Zürich |
| Kongresse | Perplexity | Relevante Kongresse mit >1.000 Teilnehmern |
| Fussball | Transfermarkt.de | Heimspielkalender FCZ und GCZ je Saison |

**Ordnerstruktur:**
- `data/raw/events/` — Einzelne Quelldateien, unverändert
- `data/interim/events.csv` — Zusammengeführt, gefiltert, bereinigt


## Feiertage

Gesetzliche und nicht-gesetzliche Feiertage des Kantons Zürich.
Generiert über das Python-Package `holidays`, ergänzt um lokale Feiertage
(Berchtoldstag, Sechseläuten, Knabenschiessen).


## Stadtfeste

Grosse Zürcher Stadtfeste mit >1.000 Besuchern, recherchiert über Gemini.
Erfasste Events: Züri Fäscht, Street Parade, Silvesterzauber.


## Konzerte

Grosskonzerte im Stadion Letzigrund (>1.000 Besucher), recherchiert über Perplexity.


## Fachmessen & Kongresse

Grosse Messen und Kongresse in Zürich, recherchiert über Perplexity.
Locations: Messe Zürich, Hallenstadion, StageOne Zürich-Oerlikon, Kongresshaus.


## Fussball

Heimspiele der Zürcher Fussballclubs, manuell erfasst über Transfermarkt.de.
Erfasste Vereine: FC Zürich (Super League, UEFA), Grasshopper Club Zürich (Super League).

> Einzelne Spieltage können durch Verlegungen abweichen und sollten
> vor der Modellierung gegen die Originaldaten geprüft werden.


## Master-Kalender

Alle Kategorien zusammengeführt, auf 2023–2025 eingeschränkt, nach Datum sortiert.

## Feiertage

Gesetzliche Feiertage des Kantons Zürich über das Python-Package `holidays` automatisch generiert.  
Nicht-gesetzliche Feiertage (z.B. Berchtoldstag) wurden über Gemini recherchiert und ergänzt.

In [9]:
import pandas as pd

PATH = '../../data/raw/vbz/events/'

In [ ]:
# HOLIDAYS 

df_holidays = pd.read_csv(PATH+"holidays.csv", sep=";")
df_holidays["Datum"] = pd.to_datetime(df_holidays["Datum"])
print(df_holidays.info())
print(df_holidays.describe(include="all"))
df_holidays.head()

<class 'pandas.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Datum       36 non-null     datetime64[us]
 1   Event_Name  36 non-null     str           
 2   Typ         36 non-null     str           
 3   Gewichtung  36 non-null     float64       
 4   Ort         36 non-null     str           
dtypes: datetime64[us](1), float64(1), str(3)
memory usage: 2.5 KB
None
                      Datum   Event_Name       Typ  Gewichtung     Ort
count                    36           36        36        36.0      36
unique                  NaN           12         1         NaN       1
top                     NaN  Neujahrstag  Feiertag         NaN  Zürich
freq                    NaN            3        36         NaN      36
mean    2024-06-07 08:40:00          NaN       NaN         1.0     NaN
min     2023-01-01 00:00:00          NaN       NaN         1.0     NaN
25% 

,Datum,Event_Name,Typ,Gewichtung,Ort
0,2023-01-01,Neujahrstag,Feiertag,1.0,Zürich
1,2023-01-02,Berchtoldstag,Feiertag,1.0,Zürich
2,2023-04-07,Karfreitag,Feiertag,1.0,Zürich
3,2023-04-10,Ostermontag,Feiertag,1.0,Zürich
4,2023-04-17,Sechseläuten,Feiertag,1.0,Zürich


In [12]:
# PARTIES

df_parties = pd.read_csv(PATH+"parties.csv", sep=";")
df_parties["Datum"] = pd.to_datetime(df_parties["Datum"])
print(df_parties.info())
print(df_parties.describe(include="all"))
df_parties.head()

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Datum       12 non-null     datetime64[us]
 1   Event_Name  12 non-null     str           
 2   Typ         12 non-null     str           
 3   Gewichtung  12 non-null     int64         
 4   Ort         12 non-null     str           
dtypes: datetime64[us](1), int64(1), str(3)
memory usage: 972.0 bytes
None
                      Datum       Event_Name        Typ  Gewichtung     Ort
count                    12               12         12   12.000000      12
unique                  NaN                3          1         NaN       1
top                     NaN  Silvesterzauber  Stadtfest         NaN  Zürich
freq                    NaN                6         12         NaN      12
mean    2024-07-12 20:00:00              NaN        NaN    2.500000     NaN
min     2023-07-07 00:00:00              N

,Datum,Event_Name,Typ,Gewichtung,Ort
0,2023-07-07,Züri Fäscht,Stadtfest,3,Zürich
1,2023-07-08,Züri Fäscht,Stadtfest,3,Zürich
2,2023-07-09,Züri Fäscht,Stadtfest,3,Zürich
3,2023-08-12,Street Parade,Stadtfest,3,Zürich
4,2023-12-31,Silvesterzauber,Stadtfest,2,Zürich


In [13]:
# CONCERTS

df_concerts = pd.read_csv(PATH+"concerts.csv", sep=";")
df_concerts["Datum"] = pd.to_datetime(df_concerts["Datum"])
print(df_concerts.info())
print(df_concerts.describe(include="all"))
df_concerts.head()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Datum       5 non-null      datetime64[us]
 1   Event_Name  5 non-null      str           
 2   Typ         5 non-null      str           
 3   Gewichtung  5 non-null      int64         
 4   Ort         5 non-null      str           
dtypes: datetime64[us](1), int64(1), str(3)
memory usage: 506.0 bytes
None
                      Datum    Event_Name      Typ  Gewichtung  \
count                     5             5        5    5.000000   
unique                  NaN             3        1         NaN   
top                     NaN  Taylor Swift  Konzert         NaN   
freq                    NaN             2        5         NaN   
mean    2024-12-10 00:00:00           NaN      NaN    2.600000   
min     2024-06-29 00:00:00           NaN      NaN    2.000000   
25%     2024-07-09 00:00:00           

,Datum,Event_Name,Typ,Gewichtung,Ort
0,2024-06-29,AC/DC,Konzert,3,Stadion Letzigrund
1,2024-07-09,Taylor Swift,Konzert,3,Stadion Letzigrund
2,2024-07-10,Taylor Swift,Konzert,3,Stadion Letzigrund
3,2025-08-02,Ed Sheeran,Konzert,2,Stadion Letzigrund
4,2025-08-03,Ed Sheeran,Konzert,2,Stadion Letzigrund


In [14]:
# FAIRS

df_fairs = pd.read_csv(PATH+"fairs.csv", sep=";")
df_fairs["Datum"] = pd.to_datetime(df_fairs["Datum"])
print(df_fairs.info())
print(df_fairs.describe(include="all"))
df_fairs.head()


<class 'pandas.DataFrame'>
RangeIndex: 83 entries, 0 to 82
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Datum       83 non-null     datetime64[us]
 1   Event_Name  83 non-null     str           
 2   Typ         83 non-null     str           
 3   Gewichtung  83 non-null     int64         
 4   Ort         83 non-null     str           
dtypes: datetime64[us](1), int64(1), str(3)
memory usage: 6.9 KB
None
                             Datum           Event_Name        Typ  \
count                           83                   83         83   
unique                         NaN                   26          2   
top                            NaN  Zürich Design Weeks  Fachmesse   
freq                           NaN                   11         54   
mean    2024-10-20 14:10:07.228915                  NaN        NaN   
min            2023-01-13 00:00:00                  NaN        NaN   
25%          

,Datum,Event_Name,Typ,Gewichtung,Ort
0,2023-01-13,World Crypto Conference,Kongress,1,Zürich
1,2023-01-14,World Crypto Conference,Kongress,1,Zürich
2,2023-01-15,World Crypto Conference,Kongress,1,Zürich
3,2023-03-28,Personal Swiss,Fachmesse,1,Messe Zürich
4,2023-03-29,Personal Swiss,Fachmesse,1,Messe Zürich


In [15]:
# SOCCER

df_soccer = pd.read_csv(PATH+"soccer.csv", sep=";")
df_soccer["Datum"] = pd.to_datetime(df_soccer["Datum"])
print(df_soccer.info())
print(df_soccer.describe(include="all"))
df_soccer.head()


<class 'pandas.DataFrame'>
RangeIndex: 165 entries, 0 to 164
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Datum       165 non-null    datetime64[us]
 1   Event_Name  165 non-null    str           
 2   Typ         165 non-null    str           
 3   Gewichtung  165 non-null    int64         
 4   Ort         165 non-null    str           
dtypes: datetime64[us](1), int64(1), str(3)
memory usage: 13.5 KB
None
                             Datum            Event_Name           Typ  \
count                          165                   165           165   
unique                         NaN                    35             7   
top                            NaN  FCZ vs FC Winterthur  Super League   
freq                           NaN                     9           150   
mean    2024-06-19 07:51:16.363636                   NaN           NaN   
min            2022-07-23 00:00:00                   NaN

,Datum,Event_Name,Typ,Gewichtung,Ort
0,2022-07-23,FCZ vs FC Luzern,Super League,2,Letzigrund
1,2022-07-24,GCZ vs FC Lugano,Super League,1,Letzigrund
2,2022-07-27,FCZ vs Qarabağ Ağdam,UEFA Champions League Qual.,2,Letzigrund
3,2022-08-06,GCZ vs FC St. Gallen 1879,Super League,1,Letzigrund
4,2022-08-07,FCZ vs FC Sion,Super League,2,Letzigrund


In [16]:
# TOTAL EVENTS

# Master zusammenführen
sources = [df_holidays, df_parties, df_concerts, df_fairs, df_soccer]
df_events = pd.concat(sources, ignore_index=True)
df_events["Datum"] = pd.to_datetime(df_events["Datum"])

df_events = df_events[
    (df_events["Datum"] >= "2023-01-01") &
    (df_events["Datum"] <= "2025-12-31")
].sort_values("Datum").reset_index(drop=True)

df_events.to_csv("../../data/interim/vbz/events.csv", index=False, sep=";")

print(df_events.info())
print(df_events.describe(include="all"))
print(df_events.groupby(["Typ", "Gewichtung"]).size().rename("Anzahl"))
df_events.head(10)


<class 'pandas.DataFrame'>
RangeIndex: 258 entries, 0 to 257
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Datum       258 non-null    datetime64[us]
 1   Event_Name  258 non-null    str           
 2   Typ         258 non-null    str           
 3   Gewichtung  258 non-null    float64       
 4   Ort         258 non-null    str           
dtypes: datetime64[us](1), float64(1), str(3)
memory usage: 20.5 KB
None
                             Datum           Event_Name           Typ  \
count                          258                  258           258   
unique                         NaN                   73             9   
top                            NaN  Zürich Design Weeks  Super League   
freq                           NaN                   11           115   
mean    2024-08-04 05:46:02.790697                  NaN           NaN   
min            2023-01-01 00:00:00                  NaN     

,Datum,Event_Name,Typ,Gewichtung,Ort
0,2023-01-01,Neujahrstag,Feiertag,1.0,Zürich
1,2023-01-02,Berchtoldstag,Feiertag,1.0,Zürich
2,2023-01-13,World Crypto Conference,Kongress,1.0,Zürich
3,2023-01-14,World Crypto Conference,Kongress,1.0,Zürich
4,2023-01-15,World Crypto Conference,Kongress,1.0,Zürich
5,2023-01-21,GCZ vs BSC Young Boys,Super League,1.0,Letzigrund
6,2023-01-29,FCZ vs FC St. Gallen 1879,Super League,2.0,Letzigrund
7,2023-02-01,GCZ vs FC Basel 1893,Schweizer Cup,1.0,Letzigrund
8,2023-02-04,GCZ vs FC Basel 1893,Super League,1.0,Letzigrund
9,2023-02-12,FCZ vs FC Winterthur,Super League,2.0,Letzigrund
